In [ ]:
import pandas as pd

# generate by ChatGPT, with some manual edits
with open("data_asg1.txt", encoding="utf-8") as f:
    texts = f.readlines()

with open("labels_asg1.txt") as f:
    labels = f.read().split()

df = pd.DataFrame({
    "text": texts,
    "label": labels
})

df.head()


,text,label
0,seeing ppl walking w/ crutches makes me really...,1
1,"look for the girl with the broken smile, ask h...",0
2,Now I remember why I buy books online @user #s...,1
3,@user @user So is he banded from wearing the c...,1
4,Just found out there are Etch A Sketch apps. ...,1


In [ ]:
# 1)Frequency-based embeddings

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score, f1_score
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.feature_selection import SelectKBest, chi2
from sklearn.svm import LinearSVC
from sklearn.metrics import classification_report

X = df["text"].values
y = df["label"].values

acc_scores = []
f1_scores = []
macro_f1_scores = []

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

for fold, (train_idx, test_idx) in enumerate(skf.split(X, y), 1):
    X_train, X_test = X[train_idx], X[test_idx]
    y_train, y_test = y[train_idx], y[test_idx]

    vectorizer = TfidfVectorizer()
    X_train_vec = vectorizer.fit_transform(X_train)
    X_test_vec = vectorizer.transform(X_test)

    selector = SelectKBest(chi2, k=100)
    X_train_vec_kbest = selector.fit_transform(X_train_vec, y_train)
    X_test_vec_kbest = selector.transform(X_test_vec)
    
    # train classifier
    clf = LinearSVC()
    clf.fit(X_train_vec_kbest, y_train)

    # predict
    y_pred = clf.predict(X_test_vec_kbest)

    # compute metrics
    acc_scores.append(accuracy_score(y_test, y_pred))
    f1_scores.append(f1_score(y_test, y_pred, pos_label="1"))
    macro_f1_scores.append(f1_score(y_test, y_pred, average='macro'))

    print(f"Fold {fold} - acc: {acc_scores[-1]:.4f}, f1: {f1_scores[-1]:.4f}, macro_f1: {macro_f1_scores[-1]:.4f}")

print("Mean accuracy:", sum(acc_scores) / len(acc_scores))
print("Mean F1:", sum(f1_scores) / len(f1_scores))
print("Mean macro-F1:", sum(macro_f1_scores) / len(macro_f1_scores))

Fold 1 - acc: 0.6558, f1: 0.5710, macro_f1: 0.6418
Fold 2 - acc: 0.6533, f1: 0.5752, macro_f1: 0.6412
Fold 3 - acc: 0.6630, f1: 0.5889, macro_f1: 0.6517
Fold 4 - acc: 0.6217, f1: 0.5573, macro_f1: 0.6135
Fold 5 - acc: 0.6609, f1: 0.6071, macro_f1: 0.6544
Mean accuracy: 0.6509443893688335
Mean F1: 0.5798878385567313
Mean macro-F1: 0.6405210760672823


In [ ]:
# Compare different classifiers with TFIDF features
from sklearn.svm import LinearSVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier

def cv_scores_model(X, y, model):
    accs, f1s, macro_f1s = [], [], []
    for train_idx, test_idx in skf.split(X, y):
        X_train, X_test = X[train_idx], X[test_idx]
        y_train, y_test = y[train_idx], y[test_idx]

        clf = model()
        clf.fit(X_train, y_train)
        y_pred = clf.predict(X_test)

        accs.append(accuracy_score(y_test, y_pred))
        f1s.append(f1_score(y_test, y_pred, pos_label="1"))
        macro_f1s.append(f1_score(y_test, y_pred, average="macro"))
    return np.mean(accs), np.mean(f1s), np.mean(macro_f1s)

print("TFIDF + LinearSVC:", cv_scores_model(X_tfidf, y, LinearSVC))
print("TFIDF + RandomForest:", cv_scores_model(X_tfidf, y, RandomForestClassifier))
print("TFIDF + DecisionTree:", cv_scores_model(X_tfidf, y, DecisionTreeClassifier))


NameError: name 'X_tfidf' is not defined